# Label Smoothing Loss

**难度:** Easy

实现标签平滑交叉熵损失。

标签平滑通过将 one-hot 目标分布与均匀分布混合来防止过度自信，在 Transformer 训练中被广泛使用。

**签名:** `label_smoothing(logits, targets, smoothing=0.1) -> Tensor`

**参数:**
- `logits` — 模型原始输出，形状 `(N, C)`
- `targets` — 整数类别索引，形状 `(N,)`
- `smoothing` — 平滑系数 ε ∈ [0, 1)

**返回:** 标量损失

**公式:** 正确类别的软目标 = `1 - ε`，其他类别 = `ε / (C - 1)`

**提示:**
1. `soft = full((N,C), ε/(C-1))`，再 `soft.scatter_(1, targets, 1-ε)`
2. 用稳定 log-softmax 得到 `log_probs`
3. `return -(soft * log_probs).sum(dim=-1).mean()`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ SOLUTION

def label_smoothing(logits, targets, smoothing=0.1):
    N, C = logits.shape

    # ---- Step 1: 计算 log softmax ----
    # 使用 F.log_softmax 而非 log(softmax()) 是数值稳定技巧
    # 内部实现：log_softmax(x) = x - log(sum(exp(x)))，用 log-sum-exp 技巧避免溢出
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # ---- Step 2: 构造软目标分布 ----
    # 软目标 = (1-ε) * one_hot(target) + ε * uniform
    # 均匀分布部分：每个类别概率 = 1/C
    # 但正确类别要排除，所以其他类别 = ε / (C-1)
    # 先填充所有位置为 ε/(C-1)
    soft_targets = torch.full_like(log_probs, smoothing / (C - 1))

    # 在目标位置写入 1-ε（覆盖掉之前的 ε/(C-1)）
    # scatter_(dim, index, src)：在 dim 维度上，按 index 指定的位置写入 src
    # targets.unsqueeze(1): (N,) → (N,1)，用于索引 dim=1
    soft_targets.scatter_(1, targets.unsqueeze(1), 1.0 - smoothing)

    # ---- Step 3: 计算交叉熵损失 ----
    # 交叉熵 = -Σ soft_targets * log_probs
    # 这等价于 KL(soft_targets || model_probs) + H(soft_targets)
    # H(soft_targets) 是常数，优化时可以忽略
    # sum(dim=-1): 对每个样本的 C 个类别求和 → (N,)
    # mean(): 对 N 个样本取平均 → scalar
    return -(soft_targets * log_probs).sum(dim=-1).mean()

In [ ]:
from torch_judge import check
check("label_smoothing")

## 📝 核心思路总结

1. **标签平滑 = 混合 one-hot 和均匀分布**：防止模型对正确类别过度自信（overconfidence）
2. **scatter_ 实现高效赋值**：在目标位置写入 1-ε，其余位置已经是 ε/(C-1)
3. **交叉熵 = -Σ p·log(q)**：用软目标 p 和模型 log 概率 q 计算 KL 散度